In [ ]:
from tqdm import tqdm
import numpy as np
import warnings
import os
import os.path as osp
from autoXRD.spectrum_generation import strain_shifts, uniform_shifts, intensity_changes, peak_broadening
from pymatgen.core import Structure
import time
from multiprocessing import Pool
import shutil
import random
warnings.filterwarnings("ignore")

# Set Basic Parameters

In [ ]:
max_texture = 0.5 # default: texture associated with up to +/- 50% changes in peak intensities
min_domain_size, max_domain_size = 5.0, 30.0 # default: domain sizes ranging from 5 to 30 nm
max_strain = 0.03 # default: up to +/- 3% strain
max_shift = 0.5 # default: up to +/- 0.5 degrees shift in two-theta
impur_amt = 70.0 # Max amount of impurity phases to include (%)
num_spectra = 250 # Number of spectra to simulate per phase
separate = False # If False: apply all artifacts simultaneously
min_angle, max_angle = 10.0, 80.0

# Generateing Augmented Patterns

In [ ]:
import pickle
from preprocess import main
from mixed import MixedGen

saveroot = '../save/data'
for dataname in ['zeolite','limnti']:
    data_dir = osp.join(saveroot,'%s'%(dataname))
    cif_dir = osp.join(data_dir,'cif')
    aug_dir = osp.join(saveroot,'%s'%(dataname),'aug_signal')
    if not os.path.exists(aug_dir):
        os.makedirs(aug_dir)

    print('processing')

    fns = os.listdir(cif_dir)
    for fn in fns:
        if not osp.exists(osp.join(aug_dir,fn.split('.')[0]+'.npy')):
            
            mixed_gen = MixedGen(fn,max_shift=max_shift,max_texture=max_texture,max_strain=max_strain,min_angle=min_angle,max_angle=max_angle,min_domain_size=min_domain_size,max_domain_size=max_domain_size,impur_amt=impur_amt,ref_dir=cif_dir)

            patterns = [mixed_gen.mixed_pdf for i in tqdm(range(num_spectra))]
            with open(osp.join(aug_dir,fn.split('.')[0]+'.npy'),'wb') as f:
                pickle.dump(patterns,f)

    print('done!')